# Bootstrap CI para o resultado da Etapa 4 (ITUB4)

Recomputa o Transformer + FinBERT da Etapa 4 com **a mesma arquitetura, mesmo split, mesma seed**, e reporta:

- AUC pontual (deve ficar próximo de 0.709)
- **Intervalo de confiança bootstrap 95%** sobre 1.000 reamostragens do conjunto de teste

## Por que isto importa

O dumb baseline (XGBoost, 5 features autoregressivas, sem sentimento) deu **AUC = 0.658 [0.565, 0.744]**.

A pergunta crítica: o intervalo de confiança da Etapa 4 **se sobrepõe** ao intervalo do baseline? Se sim, a diferença não é estatisticamente significativa e a tese precisa ser reescrita com mais cuidado.

In [1]:
import sys, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from eval_utils import (
    walk_forward_split, evaluate_model, make_binary_target,
)

PRICE_PATH = '../2.stocks/dataset_full.csv'
SENTIMENT_PATH = '../4.finbert-br/itub4_daily_sentiment.csv'
HORIZON = 21
WINDOW = 30
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

## 1. Construir o dataset Etapa 4: 11 preço + 5 sentimento

In [2]:
PRICE_COLS = ['Close','Volume','return','ma7','ma21','std21','lag_1','lag_2','lag_3','lag_4','lag_5']
SENT_COLS = ['n_articles','mean_logit_pos','mean_logit_neg','mean_logit_neu','mean_sentiment']

px = pd.read_csv(PRICE_PATH, parse_dates=['Date']).rename(columns={'Date':'date'})
px = px[['date'] + PRICE_COLS].sort_values('date').reset_index(drop=True)

sent = pd.read_csv(SENTIMENT_PATH, parse_dates=['date'])[['date'] + SENT_COLS]
df = px.merge(sent, on='date', how='left').sort_values('date').reset_index(drop=True)
df[SENT_COLS] = df[SENT_COLS].ffill().fillna(0)

df['target'] = make_binary_target(df['Close'], horizon=HORIZON)
df = df.dropna(subset=['target']).reset_index(drop=True)
FEATURES = PRICE_COLS + SENT_COLS
print(f'{len(df)} dias × {len(FEATURES)} features | balance={df["target"].mean():.3f}')

1206 dias × 16 features | balance=0.590


In [3]:
train_df, val_df, test_df = walk_forward_split(df)
scaler = StandardScaler().fit(train_df[FEATURES])
X_train = scaler.transform(train_df[FEATURES]); y_train = train_df['target'].values.astype(int)
X_val   = scaler.transform(val_df[FEATURES]);   y_val   = val_df['target'].values.astype(int)
X_test  = scaler.transform(test_df[FEATURES]);  y_test  = test_df['target'].values.astype(int)

def make_windows(X, y, window=WINDOW):
    Xs, ys = [], []
    for i in range(window, len(X)):
        Xs.append(X[i-window:i]); ys.append(y[i])
    return np.array(Xs), np.array(ys)

Xt, yt = make_windows(X_train, y_train)
Xv, yv = make_windows(X_val, y_val)
Xte, yte = make_windows(X_test, y_test)
print('Shapes:', Xt.shape, Xv.shape, Xte.shape)

Shapes: (814, 30, 16) (150, 30, 16) (152, 30, 16)


## 2. Transformer da Etapa 4 (mesma arquitetura)

In [4]:
class Stage4Transformer(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, nlayers=2):
        super().__init__()
        self.proj = nn.Linear(n_features, d_model)
        # Positional encoding sinusoidal
        pe = torch.zeros(WINDOW, d_model)
        pos = torch.arange(0, WINDOW).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos*div); pe[:, 1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=128, dropout=0.2, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=nlayers)
        self.head = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 1))
    def forward(self, x):
        h = self.proj(x) + self.pe[:, :x.size(1), :]
        h = self.enc(h)
        return self.head(h.mean(dim=1)).squeeze(-1)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Stage4Transformer(n_features=len(FEATURES)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
pos = (yt==1).sum(); neg = (yt==0).sum()
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg/max(pos,1)], device=device, dtype=torch.float32))

Xt_t = torch.tensor(Xt, dtype=torch.float32, device=device)
yt_t = torch.tensor(yt, dtype=torch.float32, device=device)
Xv_t = torch.tensor(Xv, dtype=torch.float32, device=device)
yv_t = torch.tensor(yv, dtype=torch.float32, device=device)
Xte_t = torch.tensor(Xte, dtype=torch.float32, device=device)

best=float('inf'); best_state=None; bad=0; patience=15
for epoch in range(300):
    model.train(); opt.zero_grad()
    loss = loss_fn(model(Xt_t), yt_t); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    model.eval()
    with torch.no_grad():
        vl = loss_fn(model(Xv_t), yv_t).item()
    if vl < best - 1e-4:
        best=vl; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
    else:
        bad += 1
        if bad >= patience: break
model.load_state_dict(best_state); model.eval()
with torch.no_grad():
    y_score = torch.sigmoid(model(Xte_t)).cpu().numpy()

## 3. AUC + bootstrap CI

In [5]:
results = evaluate_model(yte, y_score, n_boot=1000)
print('Stage 4 Transformer + FinBERT (ITUB4)')
print('  AUC:        ', results['auc_str'])
print('  Accuracy:   ', f"{results['accuracy']:.3f}")
print('  F1(Sobe):   ', f"{results['f1_pos']:.3f}")
print('  F1(Desce):  ', f"{results['f1_neg']:.3f}")
print('  Confusion:  ', results['confusion'])

# Comparação com o dumb baseline
BASELINE = (0.658, 0.565, 0.744)
print('\nComparação com dumb baseline:')
print(f'  Baseline AUC: {BASELINE[0]:.3f} [{BASELINE[1]:.3f}, {BASELINE[2]:.3f}]')
print(f'  Stage 4 AUC:  {results["auc_str"]}')
lo4, hi4 = results['auc_ci']
overlap = not (hi4 < BASELINE[1] or lo4 > BASELINE[2])
print(f'  CIs sobrepõem? {"SIM — diferença NÃO é estatisticamente significativa" if overlap else "NÃO — diferença é estatisticamente significativa"}')

Stage 4 Transformer + FinBERT (ITUB4)
  AUC:         0.442 [0.358, 0.528]
  Accuracy:    0.237
  F1(Sobe):    0.000
  F1(Desce):   0.383
  Confusion:   [[36, 0], [116, 0]]

Comparação com dumb baseline:
  Baseline AUC: 0.658 [0.565, 0.744]
  Stage 4 AUC:  0.442 [0.358, 0.528]
  CIs sobrepõem? NÃO — diferença é estatisticamente significativa


## 4. Interpretação

Se o intervalo se sobrepõe ao do dumb baseline (0.565–0.744), o resultado de 0.709 da Etapa 4 não pode ser apresentado como superioridade — apenas como **consistente com** o baseline.

Nesse caso, a tese precisa ser reescrita ao redor de **comparação interna** (Etapa 3 vs Etapa 4: mesmo modelo, representações diferentes), não ao redor de **previsão absoluta**.

Próximo passo: rodar `multi_ticker.ipynb` para ver se o achado se replica em PETR4 e VALE3 — três tickers com gap consistente é evidência muito mais forte que um único.